# BayesBreak quickstart

BayesBreak is a scikit-learn compatible Bayesian segmenter. Each estimator takes `fit(X, y)`, exposes `predict(X)`, `score(X, y)`, `transform(X)`, and a rich set of fitted attributes (`k_map_`, `map_boundaries_`, `k_posterior_`, `boundary_marginals_`, `bayes_curve_mean_`, `log_evidence_`, ...).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from bayesbreak import BayesBreakBernoulli, BayesBreakGaussian, BayesBreakPoisson

rng = np.random.default_rng(0)
true_means = [0.0, 2.5, -1.0]
segment_lengths = [80, 60, 60]
y = np.concatenate([rng.normal(mu, 0.3, L) for mu, L in zip(true_means, segment_lengths)])
X = np.arange(y.size).reshape(-1, 1)

model = BayesBreakGaussian(k_max=10, regression_curve='mix_k').fit(X, y)
print('k_map         :', model.k_map_)
print('MAP boundaries:', model.map_boundaries_)
print('log p(y)      :', model.log_evidence_)
print('score (y|y)   :', model.score(X, y))

### Posterior over segment count `P(k | y)`

In [ ]:
fig, ax = plt.subplots(figsize=(5, 2.5))
k_values = np.arange(1, model.k_posterior_.size + 1)
ax.bar(k_values, model.k_posterior_, color='#4477AA')
ax.axvline(model.k_map_, color='#EE6677', ls='--', label=f'k_map={model.k_map_}')
ax.set_xlabel('k')
ax.set_ylabel('P(k | y)')
ax.legend()
plt.show()

### Boundary marginals `P(b_i = 1 | y)`

In [ ]:
fig, ax = plt.subplots(figsize=(8, 2.5))
ax.bar(np.arange(1, y.size), model.boundary_marginals_, color='#228833')
for b in model.map_boundaries_[1:-1]:
    ax.axvline(b, color='#EE6677', ls=':')
ax.set_xlabel('interior index')
ax.set_ylabel('P(b_i = 1 | y)')
plt.show()

### MAP piecewise-constant fit and Bayesian regression curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(y, 'o', ms=2, alpha=0.4, color='#888', label='y')
ax.plot(model.predict(X), color='#EE6677', lw=2, label='MAP')
ax.plot(model.bayes_curve_mean_, color='#4477AA', lw=1.5, label='Bayes curve')
ax.legend()
ax.set_xlabel('index')
plt.show()

### Other families

Every family shares the same API.

In [ ]:
rates = [2.0, 8.0, 3.0]
y_p = np.concatenate([rng.poisson(r, L) for r, L in zip(rates, [50, 50, 50])])
X_p = np.arange(y_p.size).reshape(-1, 1)
m_p = BayesBreakPoisson(k_max=8).fit(X_p, y_p)
print('Poisson MAP boundaries:', m_p.map_boundaries_)

probs = [0.2, 0.8, 0.3]
y_b = np.concatenate([rng.binomial(1, p, L) for p, L in zip(probs, [50, 50, 50])])
X_b = np.arange(y_b.size).reshape(-1, 1)
m_b = BayesBreakBernoulli(k_max=8).fit(X_b, y_b)
print('Bernoulli MAP boundaries:', m_b.map_boundaries_)

## Pipeline + cross-validation

BayesBreak works inside `sklearn.pipeline.Pipeline`; `score` returns the mean posterior-predictive log-density and is CV-compatible.

In [ ]:
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

pipe = Pipeline([('scale', StandardScaler()), ('seg', BayesBreakGaussian(k_max=8))])
cv_scores = cross_val_score(pipe, X, y, cv=TimeSeriesSplit(3))
print('CV scores:', cv_scores)